# EDA v2 — Costos de importación Hortifrut Perú
**Notebook de presentación.** Los cálculos viven en `src/eda/00…07_*.py` (fuente de verdad, reproducibles con `python src/run_all.py`); aquí se cargan los resultados ya generados y se muestran tablas y figuras clave.

Metodología CRISP-DM · target validado = **servicios logísticos** (tributos aparte) sobre **target fiable**.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('../src/lib'))
import pandas as pd, numpy as np
from IPython.display import Image, Markdown, display
import dataset as D, config as C
FIG = '../reports/eda/figures'
df = D.load(fiable_only=True)
print('Operaciones fiables:', df.shape)

## 4.0 Inventario y armonización
8 CSV (1 expense + 7 operativos) con deriva de esquema español→inglés. Ver `reports/eda/00_inventario.md` y `reports/eda/02_armonizacion_columnas.md`.

In [ ]:
op = pd.read_parquet('../data_csv/processed/operativo_lineas.parquet')
exp = pd.read_parquet('../data_csv/processed/expense_lineas.parquet')
print('operativo_lineas:', op.shape, '| expense_lineas:', exp.shape)
print('esquemas:', sorted(op["schema"].unique()))

## 4.0.5 / 4.1 Conceptos canónicos y reconstrucción del target
81 conceptos → 14 canónicos; join por `op_id_full` (0 colisiones, 98.4%).

In [ ]:
g = (exp.assign(cc=exp['concepto_raw'].map(__import__('utils').canon_concept))
        .groupby('cc')['monto_usd'].agg(['size','sum']).sort_values('sum', ascending=False))
g['usd_pct'] = (100*g['sum']/g['sum'].sum()).round(1); display(g.round(0))
display(df[['target_servicios','tributos_usd','target_total']].describe().round(0))

In [ ]:
Image(f'{FIG}/f02_target_dist.png')

In [ ]:
Image(f'{FIG}/f02_conceptos.png')

## 4.3 / 4.4 Distribución del target y drivers
Target asimétrico (skew≈9) → log. Drivers por ε² (Kruskal-Wallis).

In [ ]:
Image(f'{FIG}/f04_target.png')

In [ ]:
Image(f'{FIG}/f05_drivers_cat.png')

In [ ]:
Image(f'{FIG}/f05_escala.png')

In [ ]:
Image(f'{FIG}/f05_corr.png')

## 4.5 Tiempos del proceso y estacionalidad
Más días en depósito ⇒ más sobrestadía (ρ=0.53). Deriva temporal ≈ +15%/año ⇒ validación temporal.

In [ ]:
Image(f'{FIG}/f06_temporal.png')

## 4.6 Segmentos, clustering e importancia preliminar
SUSTRATOS×SEA = 46% del gasto. Baseline LightGBM (split temporal): MdAPE 24.9%.

In [ ]:
Image(f'{FIG}/f07_segmentos.png')

In [ ]:
Image(f'{FIG}/f07_clusters.png')

In [ ]:
Image(f'{FIG}/f07_importancia.png')

## 4.7 Diagnóstico y entregables
- Diccionario: `reports/eda/05_diccionario_datos.md`
- Features candidatas: `reports/eda/11_features_candidatas.md`
- Tabla de decisiones: `reports/eda/12_tabla_decisiones.md`
- **Hallazgos + 9 preguntas:** `reports/eda/13_hallazgos.md`

In [ ]:
print(open('../reports/eda/13_hallazgos.md', encoding='utf-8').read()[:1500])